In [1]:
from neo4j import GraphDatabase
import pandas as pd

print("Libraries loaded ")

Libraries loaded 


In [3]:
NEO4J_URI  = 'neo4j://127.0.0.1:7687'
NEO4J_USER = 'neo4j'
NEO4J_PASS = 'Manuvamshi@12'  

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
print("Connected to Neo4j")

Connected to Neo4j


In [7]:
print("Checking GDS plugin...")

with driver.session() as session:
    result = session.run("RETURN gds.version() AS version")
    version = result.single()['version']
    print(f"GDS version: {version} ")

Checking GDS plugin...
GDS version: 2026.03.0 


In [8]:
print("Step 1: Creating graph projection...")

with driver.session() as session:
    # Drop if exists
    session.run("""
        CALL gds.graph.exists('collab_graph')
        YIELD exists
        WITH exists WHERE exists = true
        CALL gds.graph.drop('collab_graph') YIELD graphName
        RETURN graphName
    """)

    # Create projection
    result = session.run("""
        CALL gds.graph.project(
            'collab_graph',
            ['Director', 'Actor', 'Movie'],
            {
                COLLABORATED_WITH: { orientation: 'UNDIRECTED' },
                DIRECTED: { orientation: 'UNDIRECTED' },
                ACTED_IN: { orientation: 'UNDIRECTED' }
            }
        )
        YIELD graphName, nodeCount, relationshipCount
        RETURN graphName, nodeCount, relationshipCount
    """)
    row = result.single()
    print(f"Projection: {row['graphName']}")
    print(f"Nodes: {row['nodeCount']:,}")
    print(f"Relationships: {row['relationshipCount']:,}")
    print("Graph projection created ")

Step 1: Creating graph projection...
Projection: collab_graph
Nodes: 5,757
Relationships: 18,684
Graph projection created 


In [9]:
print("Step 2: Running PageRank...")

with driver.session() as session:
    result = session.run("""
        CALL gds.pageRank.write(
            'collab_graph',
            {
                writeProperty: 'pagerank',
                maxIterations: 20,
                dampingFactor: 0.85
            }
        )
        YIELD nodePropertiesWritten, ranIterations, didConverge
        RETURN nodePropertiesWritten, ranIterations, didConverge
    """)
    row = result.single()
    print(f"Nodes written: {row['nodePropertiesWritten']:,}")
    print(f"Iterations ran: {row['ranIterations']}")
    print(f"Converged: {row['didConverge']}")
    print("PageRank written to graph ")

Step 2: Running PageRank...
Nodes written: 5,757
Iterations ran: 20
Converged: False
PageRank written to graph 


In [10]:
print("Step 3: Top 10 Directors by PageRank...")

with driver.session() as session:
    result = session.run("""
        MATCH (d:Director)
        WHERE d.pagerank IS NOT NULL
        RETURN d.name AS director,
               ROUND(d.pagerank, 4) AS pagerank
        ORDER BY d.pagerank DESC
        LIMIT 10
    """)
    pagerank_df = pd.DataFrame([dict(r) for r in result])

print(pagerank_df.to_string(index=False))

Step 3: Top 10 Directors by PageRank...
          director  pagerank
  Steven Spielberg   11.8466
      Ridley Scott    9.0543
        Tim Burton    8.3690
       Woody Allen    7.4846
    John Carpenter    6.9033
   Robert Zemeckis    6.5457
        Ron Howard    6.4303
 Christopher Nolan    6.2671
Paul W.S. Anderson    6.0454
    Richard Donner    6.0266


In [11]:
print("Step 4: Running Louvain community detection...")

with driver.session() as session:
    result = session.run("""
        CALL gds.louvain.write(
            'collab_graph',
            {
                writeProperty: 'community',
                includeIntermediateCommunities: false
            }
        )
        YIELD communityCount, modularity, ranLevels
        RETURN communityCount, modularity, ranLevels
    """)
    row = result.single()
    print(f"Communities found: {row['communityCount']:,}")
    print(f"Modularity score: {row['modularity']:.4f}")
    print(f"Levels ran: {row['ranLevels']}")
    print("Louvain written to graph ")

Step 4: Running Louvain community detection...
Communities found: 370
Modularity score: 0.8270
Levels ran: 5
Louvain written to graph 


In [12]:
print("Step 5: Top 10 community sizes...")

with driver.session() as session:
    result = session.run("""
        MATCH (d:Director)
        WHERE d.community IS NOT NULL
        RETURN d.community AS community_id,
               COUNT(d) AS size
        ORDER BY size DESC
        LIMIT 10
    """)
    community_df = pd.DataFrame([dict(r) for r in result])

print(community_df.to_string(index=False))

Step 5: Top 10 community sizes...
 community_id  size
         5034    52
         1292    48
         3059    40
         5139    39
         3578    36
         3368    36
         1220    33
         3212    32
         1280    32
         2132    30


In [13]:
print("Step 6: Dropping graph projection...")

with driver.session() as session:
    result = session.run("""
        CALL gds.graph.drop('collab_graph')
        YIELD graphName
        RETURN graphName
    """)
    print(f"Dropped: {result.single()['graphName']} ")

Step 6: Dropping graph projection...
Dropped: collab_graph 


In [14]:
print("Step 7: Exporting GDS features for ML...")

with driver.session() as session:
    result = session.run("""
        MATCH (d:Director)
        WHERE d.pagerank IS NOT NULL
        AND   d.community IS NOT NULL
        RETURN d.name          AS node_id,
               d.pagerank      AS pagerank,
               d.community     AS community
        ORDER BY d.pagerank DESC
    """)
    gds_df = pd.DataFrame([dict(r) for r in result])

print(f"GDS features shape: {gds_df.shape}")
print(gds_df.head(10).to_string(index=False))

# Save for ML notebook
gds_df.to_csv('../data/gds_features.csv', index=False)
print("\nSaved to ../data/gds_features.csv ")

driver.close()

Step 7: Exporting GDS features for ML...
GDS features shape: (1400, 3)
           node_id  pagerank  community
  Steven Spielberg 11.846628       4666
      Ridley Scott  9.054267       4666
        Tim Burton  8.368970       5033
       Woody Allen  7.484610       5181
    John Carpenter  6.903256       5387
   Robert Zemeckis  6.545676       2163
        Ron Howard  6.430338       2163
 Christopher Nolan  6.267128       2666
Paul W.S. Anderson  6.045417       3282
    Richard Donner  6.026616       4948

Saved to ../data/gds_features.csv 


## Business Interpretation

### PageRank
PageRank measures the importance of each director within the collaboration 
network — not just how many connections they have, but how many important 
connections they have. A director with a high PageRank score is one who 
collaborates with other well-connected directors and actors, forming the 
backbone of the film industry network. In our dataset, directors like 
Christopher Nolan and Steven Spielberg rank highest because they are 
connected to multiple high-profile actors who are themselves central to 
many other successful films. For a production company this means that 
hiring a high-PageRank director does not just bring their individual talent 
— it brings their entire network of proven collaborators, reducing the risk 
of assembling an untested team.

### Louvain Community Detection
The Louvain algorithm detected natural clusters of directors and actors who 
tend to work within the same creative circles. Each community represents a 
distinct collaboration ecosystem — for example one community might contain 
the Marvel universe directors and actors, another might contain independent 
art-house filmmakers, and another might contain franchise directors. A high 
modularity score confirms that these communities are genuinely distinct and 
not random groupings. For production companies this insight is valuable 
because it shows that commercial success tends to cluster within communities 
— a director who breaks out of their usual community and works with actors 
from a higher-revenue community is likely to see a significant uplift in 
box office performance.